In [ ]:
!cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers
!pip install -U /kaggle/working/sentence-transformers

In [ ]:
import os
import math
from typing import List, Tuple, Dict, Union
import multiprocessing
import pickle
import gc
import time
from concurrent.futures import ThreadPoolExecutor
import ctypes
from functools import partial
from pathlib import Path
from abc import abstractmethod, ABC

from tqdm.auto import tqdm
import pytorch_lightning as pl
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import (
    BatchEncoding, AutoModelForMultipleChoice,
    AutoTokenizer, PreTrainedTokenizer,
    AutoModelForSequenceClassification,
    AutoModel, PreTrainedModel,
    AutoConfig, AutoModelForCausalLM
)
from sentence_transformers import SentenceTransformer
from accelerate import init_empty_weights
from accelerate.utils.modeling import set_module_tensor_to_device
from safetensors.torch import load_file

# Hyperparameters

In [ ]:
# in retrieval ensemble MINILM_WEIGHT*minilm_similarities + (1 - MINILM_WEIGHT)*bge_small_similarities
MINILM_WEIGHT = 0.5

# Number of passages to retrieve for reranking
N_PASSAGES = 500

# Number of passages to include as context
N_FINAL_CONTEXT = 10

# Number of question characters to use in reranker:
# it gets pairs (question, passage) as input
QUESTION_MAX_LENGTH_RERANKING = 1500

# Using 512 tokens for MLM models (deberta, electra, roberta).
# These params control distribution of 512 tokens budget.
# Answer option will get 512 - MLM_MAX_PROMPT_LEN - MLM_MAX_CONTEXT_LEN
MLM_MAX_PROMPT_LEN = 60
MLM_MAX_CONTEXT_LEN = 380

# MLM ensemble
DEBERTA_WEIGHT = 0.5
ELECTRA_WEIGHT = 0.3
ROBERTA_WEIGHT = 0.2

# How many questions to give to LLMs
N_HARD = 500

# Control final ensemble
DO_XWIN = True
DO_PLATYPUS = True
PLATY_WEIGHT = 0.6
ADD_MLM_TO_ENSEMBLE = True
MLM_WEIGHT = 0.1

# Functions definition

## Queries encoding

In [ ]:
os.makedirs('outs', exist_ok=True)

In [ ]:
def clean_memory():
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)
    torch.cuda.empty_cache()

In [ ]:
def encode_questions_minilm(proc_id: int):
    """Using all-MiniLM-L6-v2 model,
    Generate embeddings for concatenation of prompt and answer options.
    It is supposed to run in 2 processes, each embeds half of the test dataframe.
    Result as float16 numpy array [len(df) x 384] is saved in pkl file.

    Args:
        proc_id (int): number of process we are in.
    """
    model = SentenceTransformer('/kaggle/input/sentence-transformers-222/all-MiniLM-L6-v2')
    df = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/test.csv')
    df = np.array_split(df, 2)[proc_id]
    texts = []
    for _, row in df.iterrows():
        text = f"{row.prompt}\n{row.A}\n{row.B}\n{row.C}\n{row.D}\n{row.E}"
        texts.append(text)
    embs = model.encode(texts, device=f'cuda:{proc_id}').astype(np.float16)
    with open(f'outs/encoded_questions_minilm_{proc_id}.pkl', 'wb') as f:
        pickle.dump(embs, f)

In [ ]:
def encode_questions_bge(proc_id):
    """Using BAAI/bge-small-en-v1.5 model,
    Generate embeddings for concatenation of prompt and answer options.
    It is supposed to run in 2 processes, each embeds half of the test dataframe.
    Result as float16 numpy array [len(df) x 384] is saved in pkl file.

    Args:
        proc_id (int): number of process we are in.
    """
    model_path = '/kaggle/input/bge-small-en-v1-5'
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path)
    model.eval()
    model.to(f'cuda:{proc_id}')

    df = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/test.csv')
    df = np.array_split(df, 2)[proc_id]
    texts = []
    instr = "Represent this sentence for searching relevant passages: "
    for _, row in df.iterrows():
        colon = ':'
        if row.prompt.endswith(':'):
            colon = ''
        text = f"{instr}{row.prompt}{colon} {row.A}; {row.B}; {row.C}; {row.D}; {row.E}."
        texts.append(text)

    embeddings = []
    dataloader = DataLoader(
        texts, batch_size=32, num_workers=0,
        collate_fn=lambda batch: tokenizer(batch, max_length=512, padding=True, truncation=True, return_tensors='pt')
    )
    with torch.no_grad():
        with torch.autocast(device_type='cuda'):
            for batch in tqdm(dataloader):
                model_output = model(**batch.to(model.device))
                sentence_embeddings = model_output[0][:, 0]
                sentence_embeddings = torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)
                embeddings.append(sentence_embeddings)
    embeddings = torch.cat(embeddings, dim=0).cpu().numpy().astype(np.float16)

    with open(f'outs/encoded_questions_bge_{proc_id}.pkl', 'wb') as f:
        pickle.dump(embeddings, f)

In [ ]:
def combine_embs_from_processes(model_name: str):
    """Question encoding functions produce two embeddings files for each model,
    this one combines two into one. For example:
    'outs/encoded_questions_bge_0.pkl' + 'outs/encoded_questions_bge_1.pkl' ->
    -> 'outs/encoded_questions_bge.pkl'

    Args:
        model_name (str): 'bge' or 'minilm'.
    """
    embs = []
    for proc_idx in range(2):
        with open(f'outs/encoded_questions_{model_name}_{proc_idx}.pkl', 'rb') as f:
            embs.append(pickle.load(f))
    embs = np.concatenate(embs, axis=0)
    with open(f'outs/encoded_questions_{model_name}.pkl', 'wb') as f:
        pickle.dump(embs, f)

## Retrieval

### Dense

In [ ]:
def calculate_similarities(
    model_name: str,
    passages_embs_name: str,
    device: Union[str, int]
) -> torch.Tensor:
    """Calculate dot product between questions and a portion of passages.

    Args:
        model_name (str): 'bge' or 'minilm'.
        passages_embs_name (str): name of a file in Wikipedia index.
                                  Each file contains embeddings for 1M passages, except last - it has less.
                                  Overall there are 112 files.
        device (Union[str, int]): number of gpu to use.

    Returns:
        torch.Tensor: dot prods of shape [n_questions x n_passages_in_file]
    """

    with open(f'outs/encoded_questions_{model_name}.pkl', 'rb') as f:
        query_embs = torch.tensor(pickle.load(f), dtype=torch.float16, device=device)
    passages_embs_dir = {
        'minilm': '/kaggle/input/78be3bf5/',
        'bge': '/kaggle/input/789d60d3/',
    }[model_name]
    passages_embs_path = os.path.join(passages_embs_dir, passages_embs_name)
    with open(passages_embs_path, 'rb') as f:
        passages_embs = torch.tensor(pickle.load(f), dtype=torch.float16, device=device)
    clean_memory()
    similarity = query_embs @ passages_embs.T
    return similarity

In [ ]:
def get_top_passages_ids(passages_embs_name: str):
    """Calculate similarities for both bge and minilm,
    ensemble them and get indices of the most relevant
    passages in the given chunk of passages embeddings.

    Both top similarities and top indices are saved
    in files for further concatenation of all chunks.

    Args:
        passages_embs_name (str): name of a file in Wikipedia index.
    """
    # Calculate similarities
    args = [('minilm', passages_embs_name, 'cuda:0'), ('bge', passages_embs_name, 'cuda:1')]
    with ThreadPoolExecutor() as executor:
        minilm_similarities, bge_similarities = tuple(executor.map(lambda p: calculate_similarities(*p), args))
    clean_memory()

    # Ensemble similarities of minilm and bge
    # Size of each bge_similarities and minilm_similarities for submission run is:
    # 4k * 1M * 2 bytes = 7.45 GB, so we cannot fit both into one T4 with 14.8 GB.
    # That is why doing it in batches and rewriting minilm_similarities.
    n_queries = minilm_similarities.shape[0]
    batch_size = 100
    n_batches = math.ceil(n_queries/batch_size)
    for i in range(n_batches):
        start = i*batch_size
        end = (i+1)*batch_size
        minilm_similarities[start:end] = MINILM_WEIGHT * minilm_similarities[start:end].to('cuda:0') + (1 - MINILM_WEIGHT) * bge_similarities[start:end].to('cuda:0')
    del bge_similarities
    clean_memory()

    # Sort ensembled similarities (contained in minilm_similarities tensor)
    # and get indices of most relevant passages.
    # For the same reasons as above we are doing it in batches.
    top_indices, top_similarities = [], []
    for i in range(n_batches):
        start = i*batch_size
        end = (i+1)*batch_size
        b_top_indices = torch.argsort(minilm_similarities[start:end], dim=1, descending=True)
        b_top_similarities = torch.take_along_dim(minilm_similarities[start:end], b_top_indices, dim=1)[:, :N_PASSAGES].cpu().numpy()
        b_top_indices = b_top_indices[:, :N_PASSAGES].cpu().numpy()
        top_indices.append(b_top_indices)
        top_similarities.append(b_top_similarities)
    top_similarities = np.concatenate(top_similarities, axis=0)
    top_indices = np.concatenate(top_indices, axis=0)

    # Save results
    with open('/kaggle/input/embeddings-naming/names_decode_dict.pkl', 'rb') as f:
        names_dict = pickle.load(f)

    out_name = f'outs/similarity_{names_dict[passages_embs_name]}'
    with open(out_name, 'wb') as f:
        pickle.dump(top_similarities, f)

    out_name = f'outs/indices_{names_dict[passages_embs_name]}'
    with open(out_name, 'wb') as f:
        pickle.dump(top_indices, f)

In [ ]:
def sort_files(file_names: List[str]) -> List[str]:
    """Get list of files if format 'f_start_end.pkl'
    and sort them in ascending order of start.
    Start here is the index of first passage in the chunk,
    end is the index of the last one.

    Args:
        file_names (List[str]): list of names in format 'f_start_end.pkl'.

    Returns:
        List[str]: same list, but sorted.
    """
    return sorted(file_names, key=lambda name: int(name.split('_')[1]))

def combine_ids_chunks():
    """Concatenate similarities and ids from index chunks
    and select top passages from whole wikipedia.    
    """
    
    # Read similarities from chunks and concatenate them
    with open('/kaggle/input/embeddings-naming/names_decode_dict.pkl', 'rb') as f:
        names_suffixes = list(pickle.load(f).values())
    names_suffixes = sort_files(names_suffixes)
    similarities, indices = [], []
    for suffix in names_suffixes:
        _, start, _ = suffix.split('.')[0].split('_')
        start = int(start)
        similarity_path = f'outs/similarity_{suffix}'
        with open(similarity_path, 'rb') as f:
            similarities.append(pickle.load(f))

        indices_path = f'outs/indices_{suffix}'
        with open(indices_path, 'rb') as f:
            indices.append(pickle.load(f) + start)
    similarities = np.concatenate(similarities, axis=1)
    indices = np.concatenate(indices, axis=1)
    
    # Sort similarities again and select most relevant passages
    indices_of_indices = np.argsort(-similarities, axis=1)
    final_indices = np.empty(indices.shape, dtype=np.uint32)
    for i in range(indices.shape[0]):
        for j in range(indices.shape[1]):
            final_indices[i, j] = indices[i, indices_of_indices[i, j]]
    final_indices = final_indices[:, :N_PASSAGES]

    with open('outs/passages_indices.pkl', 'wb') as f:
        pickle.dump(final_indices, f)

### Extract texts by ids

In [ ]:
def get_passages_texts_pyarrow():
    """Read saved indices of most relevant passages and
    get their texts from pyarrow index of wikipedia.

    As a result save dict with indices of passages as keys and texts as values.
    """
    with open('outs/passages_indices.pkl', 'rb') as f:
        indices = pickle.load(f)  # array n_queries x n_cands
    unique_inds = np.unique(indices)

    df_selected = pd.read_parquet('/kaggle/input/wiki-pyarrow/wiki_passages.parquet',
                                  engine='pyarrow', use_threads=True,
                                  filters=[('index', 'in', unique_inds)],
                                  columns=['index', 'passage'])
    ids2passages = dict()
    for _, row in df_selected.iterrows():
        ids2passages[row['index']] = row['passage']
    with open(f'outs/ids2passages.pkl', 'wb') as f:
        pickle.dump(ids2passages, f)

## Reranking

In [ ]:
def do_cross_encoder_scoring(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    pairs: List[Tuple[str, str]]
) -> np.array:
    """Score pairs of (question, passage) with given reranker model,
    higher score means passage is more relevant to the question.

    Args:
        model (PreTrainedModel): reranker model
        tokenizer (PreTrainedTokenizer): its tokenizer
        pairs (List[Tuple[str, str]]): List of (question, passage)

    Returns:
        np.array: array of relevance scores, shape [1 x len(pairs)]
    """
    scores = []
    dataloader = DataLoader(
        pairs, batch_size=32,
        collate_fn=lambda batch: tokenizer(
            batch, padding=True, truncation=True,
            return_tensors='pt', max_length=512)
    )
    with torch.no_grad(), torch.autocast(device_type='cuda'):
        for batch in dataloader:
            b_scores = torch.softmax(model(**batch.to(model.device)).logits, dim=-1)[:, 1].view(-1, )
            scores.append(b_scores.cpu().numpy())
    scores = np.concatenate(scores, axis=0)
    return np.expand_dims(scores, axis=0)

def rerank_passages(process_id: int, question_max_chars: int):
    """Read indices and texts of top passages from retrieval step
    and rerank them with cross-encoder model.
    Supposed to run in two processed, each on its onw half of test dataframe.
    
    Reranker scores and passages indices are saved into files.

    Args:
        process_id (int): number of the process
        question_max_chars (int): longer questions will be truncated to fit passage.
    """
    with open('outs/passages_indices.pkl', 'rb') as f:
        indices = pickle.load(f)
    with open('outs/ids2passages.pkl', 'rb') as f:
        ids2passages = pickle.load(f)
    tokenizer = AutoTokenizer.from_pretrained('/kaggle/input/ibm-reranker-tuned-2')
    model = AutoModelForSequenceClassification.from_pretrained('/kaggle/input/ibm-reranker-tuned-2')
    model.eval()
    model.to(f'cuda:{process_id}')

    scores, pas_inds = [], []
    df = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/test.csv')
    df = np.array_split(df, 2)[process_id]
    for idx, row in df.iterrows():
        text = f"{row.prompt}\n{row.A}\n{row.B}\n{row.C}\n{row.D}\n{row.E}"
        row_inds = np.unique(indices[idx])
        pas_inds.append(row_inds)
        candidates = [ids2passages[passage_id] for passage_id in row_inds]
        scores.append(do_cross_encoder_scoring(model, tokenizer, [(text[:question_max_chars], cand) for cand in candidates]))
    with open(f'outs/cross_encoder_scores_{process_id}.pkl', 'wb') as f:
        pickle.dump(scores, f)
    with open(f'outs/unique_pas_inds_reranked_{process_id}.pkl', 'wb') as f:
        pickle.dump(pas_inds, f)

In [ ]:
def build_contexts_reranked(n_cands=10):
    """Read scores and passages indices from reranker and
    build final contexts for open book inference.
    
    Result is saved into context.pkl.

    Args:
        n_cands (int, optional): Number of passages to include as context.
                                 Defaults to 10.
    """
    indices = []
    for i in range(2):
        with open(f'outs/unique_pas_inds_reranked_{i}.pkl', 'rb') as f:
            indices += pickle.load(f)
    with open('outs/ids2passages.pkl', 'rb') as f:
        id2text = pickle.load(f)
    scores = []
    for i in range(2):
        with open(f'outs/cross_encoder_scores_{i}.pkl', 'rb') as f:
            scores += pickle.load(f)
    contexts = []
    for row_idx, row in enumerate(indices):
        candidates = [id2text[passage_id] for passage_id in row]
        q_scores = scores[row_idx][0]
        sorted_inds = np.argsort(q_scores)[::-1]
        top_candidates = [candidates[cand_idx].strip() for cand_idx in sorted_inds[:n_cands]]
        contexts.append('\n'.join(top_candidates))
    with open('context.pkl', 'wb') as f:
        pickle.dump(contexts, f)

# Run code

Run each function in multiprocessing.Pool context to make sure that resources allocated for a function are released after it has finished. This helps to avoid out-of-memory errors.

## Queries encoding

In [ ]:
# MiniLM-L6-v2
start = time.time()
with multiprocessing.Pool(2) as pool:
    pool.map(encode_questions_minilm, [0, 1])
with multiprocessing.Pool(1) as pool:
    pool.map(combine_embs_from_processes, ['minilm'])
print(f'MiniLM-L6-v2 encoding: {time.time() - start} s')

In [ ]:
# bge-small
start = time.time()
with multiprocessing.Pool(2) as pool:
    pool.map(encode_questions_bge, [0, 1])
with multiprocessing.Pool(1) as pool:
    pool.map(combine_embs_from_processes, ['bge'])
print(f'bge-small encoding: {time.time() - start} s')

## Retrieval

In [ ]:
start = time.time()

file_names = os.listdir('/kaggle/input/78be3bf5')
with multiprocessing.Pool(1) as pool:
    pool.map(get_top_passages_ids, file_names)

print(f'Ids extraction: {time.time() - start} s')

In [ ]:
start = time.time()
with multiprocessing.Pool(1) as pool:
    pool.starmap(combine_ids_chunks, [()])
print(f'Similarities chunks combination: {time.time() - start} s')

In [ ]:
start = time.time()
with multiprocessing.Pool(1) as pool:
    pool.starmap(get_passages_texts_pyarrow, [()])
print(f'Text reading: {time.time() - start} s')

## Reranking

In [ ]:
start = time.time()
with multiprocessing.Pool(2) as pool:
    pool.starmap(rerank_passages, [(0, QUESTION_MAX_LENGTH_RERANKING), (1, QUESTION_MAX_LENGTH_RERANKING)])
print(f'Reranking: {time.time() - start} s')

In [ ]:
with multiprocessing.Pool(1) as pool:
    pool.map(build_contexts_reranked, [N_FINAL_CONTEXT])

In [ ]:
import shutil
if os.path.exists("outs"):
    shutil.rmtree("outs")

# Inference

## MLM

In [ ]:
class BaselineMultipleChoice(pl.LightningModule):
    def __init__(
        self,
        model_name_or_path: str,
    ):
        super().__init__()
        self.model = AutoModelForMultipleChoice.from_pretrained(model_name_or_path, local_files_only=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, local_files_only=True)

    def forward(self, batch: BatchEncoding):
        return self.model(**batch)

In [ ]:
class MultipleChoiceDataModule(pl.LightningDataModule):
    def __init__(
        self,
        proc_id: int,
        tokenizer: PreTrainedTokenizer,
        pred_file_path: str,
        max_length: int = 512,
        batch_size: int = 8,
        num_workers: int = 0,
        pin_memory: bool = False,
    ):
        """
        Data module that reads test dataframe and splits it into two halves
        to run prediction in two processes.
        Predict dataloader forms input data in the following way:
        '[CLS] Prompt [SEP] Context [SEP] Answer [SEP]'
        """
        super().__init__()
        self.save_hyperparameters(logger=False)
        self.tokenizer = tokenizer

        pred_df = pd.read_csv(pred_file_path)
        with open(f'context.pkl', 'rb') as f:
            contexts = pickle.load(f)
        pred_df['context'] = contexts
        pred_df = np.array_split(pred_df, 2)[proc_id]
        self.pred_data = list(pred_df.to_dict(orient='index').values())

        self.label2int = {
            'A': 0,
            'B': 1,
            'C': 2,
            'D': 3,
            'E': 4
        }
        self.num_choices = len(self.label2int)
        self.max_prompt_len = MLM_MAX_PROMPT_LEN
        self.max_context_len = MLM_MAX_CONTEXT_LEN

    def predict_dataloader(self):
        return DataLoader(
            self.pred_data,
            shuffle=False,
            batch_size=self.hparams.batch_size,
            drop_last=False,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            collate_fn=self._predict_collate
        )

    def _predict_collate(self, batch: List[Dict[str, str]]):
        batch_size = len(batch)
        input_ids = []
        for sample in batch:
            prompt_ids = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(sample['prompt']))
            if len(prompt_ids) > self.max_prompt_len:
                prompt_ids = prompt_ids[:self.max_prompt_len]
            context_ids = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(sample['context']))
            if len(context_ids) > self.max_context_len:
                context_ids = context_ids[:self.max_context_len]
            for cand_label in 'ABCDE':
                cand_ids = self.tokenizer(sample[cand_label], add_special_tokens=False).input_ids
                ids = (
                    [self.tokenizer.cls_token_id] + prompt_ids +
                    [self.tokenizer.sep_token_id] + context_ids +
                    [self.tokenizer.sep_token_id] + cand_ids
                )
                if len(ids) >= self.hparams.max_length:
                    ids = ids[:self.hparams.max_length - 1]
                ids += [self.tokenizer.sep_token_id]
                input_ids.append(ids)
        encoding = self.tokenizer.pad(
            {'input_ids': input_ids},
            return_tensors='pt',
            max_length=self.hparams.max_length,
            padding=True,
        )
        for key, value in encoding.items():
            encoding[key] = value.reshape(batch_size, self.num_choices, -1)
        return encoding

In [ ]:
def predict_logits(
    proc_id: int,
    test_df_path: str,
    base_model_path: str,
    ckpt_path: str,
    batch_size: int
) -> torch.Tensor:
    """Load model checkpoint, initialize data module for given process
    and run predictions.

    Args:
        proc_id (int): number of process, 0 or 1.
        test_df_path (str): path to test dataframe.
        base_model_path (str): path to base model (for example, pretrained deberta from hf)
        ckpt_path (str): path to trained checkpoint
        batch_size (int): inference batch size

    Returns:
        torch.Tensor: logits for each answer option, shape [n_questions x 5]
    """
    model = BaselineMultipleChoice.load_from_checkpoint(
        ckpt_path, model_name_or_path=base_model_path,
        map_location=f'cuda:{proc_id}', strict=False
    )
    datamodule = MultipleChoiceDataModule(
        proc_id,
        model.tokenizer,
        pred_file_path=test_df_path,
        batch_size=batch_size,
    )

    logits = []
    with torch.no_grad(), torch.autocast(device_type='cuda'):
        for batch in datamodule.predict_dataloader():
            logits.append(model(batch.to(model.device)).logits)
    logits = torch.cat(logits, dim=0).cpu()

    del model
    del datamodule
    gc.collect()

    return logits

### Deberta

In [ ]:
ckpt_dir = '/kaggle/input/deberta-v14-pretrain2/'
deberta_checkpoints = os.listdir(ckpt_dir)
deberta_checkpoints = [os.path.join(ckpt_dir, ckpt) for ckpt in deberta_checkpoints]
deberta_path = '/kaggle/input/deberta-v3-large-hf-weights'
test_df_path = '/kaggle/input/kaggle-llm-science-exam/test.csv'
batch_size = 4
arguments = [
    (test_df_path, deberta_path, ckpt, batch_size) for ckpt in deberta_checkpoints
]

In [ ]:
deberta_logits_result = []
for arg in arguments:
    with multiprocessing.Pool(2) as pool:
        logits_pair = pool.starmap(predict_logits, [(0,) + arg, (1,) + arg])
    deberta_logits_result.append(torch.cat(logits_pair, dim=0))

### Electra

In [ ]:
ckpt_dir = '/kaggle/input/electra-v14-add70k/'
electra_checkpoints = os.listdir(ckpt_dir)
electra_checkpoints = [os.path.join(ckpt_dir, ckpt) for ckpt in electra_checkpoints]
electra_path = '/kaggle/input/google-electra-large-discriminator'
test_df_path = '/kaggle/input/kaggle-llm-science-exam/test.csv'
batch_size = 4
arguments = [
    (test_df_path, electra_path, ckpt, batch_size) for ckpt in electra_checkpoints
]

In [ ]:
electra_logits_result = []
for arg in arguments:
    with multiprocessing.Pool(2) as pool:
        logits_pair = pool.starmap(predict_logits, [(0,) + arg, (1,) + arg])
    electra_logits_result.append(torch.cat(logits_pair, dim=0))

### Roberta

In [ ]:
ckpt_dir = '/kaggle/input/roberta-v14-add70k/'
roberta_checkpoints = os.listdir(ckpt_dir)
roberta_checkpoints = [os.path.join(ckpt_dir, ckpt) for ckpt in roberta_checkpoints]
roberta_path = '/kaggle/input/robertalarge'
test_df_path = '/kaggle/input/kaggle-llm-science-exam/test.csv'
batch_size = 4
arguments = [
    (test_df_path, roberta_path, ckpt, batch_size) for ckpt in roberta_checkpoints
]

In [ ]:
roberta_logits_result = []
for arg in arguments:
    with multiprocessing.Pool(2) as pool:
        logits_pair = pool.starmap(predict_logits, [(0,) + arg, (1,) + arg])
    roberta_logits_result.append(torch.cat(logits_pair, dim=0))

### Ensemble MLM logits

In [ ]:
logits = (
    DEBERTA_WEIGHT * sum(deberta_logits_result) / len(deberta_checkpoints) + 
    ELECTRA_WEIGHT * sum(electra_logits_result) / len(electra_checkpoints) +
    ROBERTA_WEIGHT * sum(roberta_logits_result) / len(roberta_checkpoints)
)
preds = torch.argsort(-logits, dim=1).numpy()

## LLM

Inference is performed for two 70b models, Platypus2 and Xwin, and is heavily relies on code from @simjeg https://www.kaggle.com/code/simjeg/platypus2-70b-with-wikipedia-rag

In [ ]:
df = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/test.csv", index_col="id")
with open('context.pkl', 'rb') as f:
    df['context'] = pickle.load(f)

# Select 'hard' questions for LLM inference.
# 'Hardest' questions are those which has the lowest score for top scored answer.
hard_questions = torch.sort(torch.softmax(logits.to(dtype=torch.float32), dim=1).max(dim=1).values).indices[:N_HARD].numpy()
df = df.iloc[hard_questions]

In [ ]:
checkpoint_path = Path("/root/.cache/")
checkpoint_path.mkdir(exist_ok=True, parents=True)

for part in [1, 2]:
    source_dir = Path(f'/kaggle/input/xwin-lm-70b-v0-1-part{part}')
    for path in source_dir.glob('*'):
        try:
            (checkpoint_path / path.name).symlink_to(path)
        except:
            pass

In [ ]:
# Class for sharded llama
MAX_LENGTH = 4096

class ShardedLlama:
    
    def __init__(self, checkpoint_path, device='cuda:0', dtype=torch.float16):
        """
        Sharded version of LlamaForCausalLM : the model is splitted into layer shards to reduce GPU memory usage.

        Parameters
        ----------
        checkpoint_path : str or Path, path to the checkpoint
        device : str, optional, by default 'cuda:0'
        dtype : torch.dtype, optional, by default torch.float16
        """
        
        # Save parameters
        self.checkpoint_path = Path(checkpoint_path)
        self.device = device 
        self.dtype = dtype

        # Create model
        self.config = AutoConfig.from_pretrained(self.checkpoint_path)
        
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = 'right'
        self.init_model()        
        self.layer_names = ['model.embed_tokens'] + [f'model.layers.{i}' for i in range(len(self.model.model.layers))] + ['model.norm', 'lm_head']
        
    def init_model(self):
                
        # Load meta model (no memory used)
        with init_empty_weights():
            self.model = AutoModelForCausalLM.from_config(self.config, trust_remote_code=True)
            self.model.tie_weights()
            
        self.layers = [self.model.model.embed_tokens] + list(self.model.model.layers) + [self.model.model.norm, self.model.lm_head]
            
        # Move buffers to device (not that much GPU memory used)
        for buffer_name, buffer in self.model.named_buffers():
            set_module_tensor_to_device(self.model, buffer_name, self.device, value=buffer, dtype=self.dtype)
       
    def load_layer(self, layer_name):
        state_dict = load_file(self.checkpoint_path / (layer_name + '.safetensors'), device=self.device)
        for param_name, param in state_dict.items():
            assert param.dtype != torch.int8, 'int8 not supported (need to add fp16_statistics)'
            set_module_tensor_to_device(self.model, param_name, self.device, value=param, dtype=self.dtype)

For convenience, create separate classes for 3 layer types with their own call method

In [ ]:
class EmbLayer(ShardedLlama):
    def __init__(self, checkpoint_path, layer_num: int, device='cuda:0', dtype=torch.float16):
        super().__init__(checkpoint_path, device, dtype)
        self.layer_names = [self.layer_names[0]]
        self.layers = [self.layers[0]]

    def __call__(self, inputs):
        # Send batch to device
        batch = [(prefix.to(self.device), suffix.to(self.device)) for prefix, suffix in inputs]
        n_suffixes = len(batch[0][1])
        suffix_eos = [(suffix != self.tokenizer.pad_token_id).sum(1) - 1 for _, suffix in inputs]

        # Create attention mask for the largest input, and position ids to use KV cache
        attention_mask = torch.finfo(self.dtype).min * torch.ones(MAX_LENGTH, MAX_LENGTH)
        attention_mask = attention_mask.triu(diagonal=1)[None, None, ...]
        attention_mask = attention_mask.to(self.device)
        position_ids = torch.arange(MAX_LENGTH, dtype=torch.long, device=self.device)[None, :]

        layer = self.layers[0]
        with torch.inference_mode():
            for j, (prefix, suffix) in enumerate(batch):
                batch[j] = (layer(prefix), layer(suffix))
        
        return batch, n_suffixes, suffix_eos, attention_mask, position_ids

In [ ]:
class MidLayer(ShardedLlama):
    def __init__(self, checkpoint_path, layer_num: int, device='cuda:0', dtype=torch.float16):
        super().__init__(checkpoint_path, device, dtype)
        assert layer_num > 0
        assert layer_num < len(self.layers) - 1
        self.layer_names = [self.layer_names[layer_num]]
        self.layers = [self.layers[layer_num]]

    def __call__(self, batch, n_suffixes, suffix_eos, attention_mask, position_ids):
        with torch.inference_mode():
            layer = self.layers[0]
            layer_name = self.layer_names[0]

            for j, (prefix, suffix) in enumerate(batch):
                if layer_name == 'model.norm':
                    # Only keep the last hidden state at this point
                    batch[j] = (None, layer(suffix[torch.arange(n_suffixes), suffix_eos[j]][:, None]))
                else:
                    # Run prefix
                    len_p, len_s = prefix.shape[1], suffix.shape[1]
                    new_prefix, (k_cache, v_cache) = layer(prefix, use_cache=True, attention_mask=attention_mask[:, :, -len_p:, -len_p:])

                    # Run suffix
                    pos = position_ids[:, len_p:len_p + len_s].repeat(n_suffixes, 1)
                    attn = attention_mask[:, :, -len_s:, -len_p - len_s:].repeat(n_suffixes, 1, 1, 1)
                    kv_cache = (k_cache.repeat(n_suffixes, 1, 1, 1), v_cache.repeat(n_suffixes, 1, 1, 1))
                    new_suffix = layer(suffix, past_key_value=kv_cache, position_ids=pos, attention_mask=attn)[0]
                    batch[j] = (new_prefix, new_suffix)
        
        return batch, n_suffixes, suffix_eos, attention_mask, position_ids

In [ ]:
class LastLayer(ShardedLlama):
    def __init__(self, checkpoint_path, layer_num: int, device='cuda:0', dtype=torch.float16):
        super().__init__(checkpoint_path, device, dtype)
        self.layer_names = [self.layer_names[-1]]
        self.layers = [self.layers[-1]]

    def __call__(self, batch, n_suffixes, suffix_eos, attention_mask, position_ids, output_token):
        with torch.inference_mode():
            layer = self.layers[0]

            for j, (prefix, suffix) in enumerate(batch):
                batch[j] = (None, layer(suffix))

        # Get scores
        batch = [suffix[:, 0, output_token].detach().cpu().numpy() for _, suffix in batch]
        
        return batch

In [ ]:
clean_memory()

In [ ]:
# Run model on the 2 GPUs
N_BATCHES = 4
MAX_CONTEXT = 2500

def get_tokens(row, tokenizer):
        system_prefix = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{instruction}\n\n### Input:\n{input_prefix}"
        instruction = f"Analyze the question and proposed answer below step by step. If the answer is correct, respond yes, otherwise respond no."
        input_prefix = f"Context: {row['context'][:MAX_CONTEXT]}\nQuestion: {row['prompt']}\nProposed answer: "
        prompt_prefix = system_prefix.format(instruction=instruction, input_prefix=input_prefix)
        prefix = tokenizer(prompt_prefix, return_tensors="pt", return_attention_mask=False, truncation=True, max_length=MAX_LENGTH)['input_ids']
        prompt_suffix = [f"{row[letter]}\n\n### Response:\n" for letter in 'ABCDE']
        suffix = tokenizer(prompt_suffix, return_tensors="pt", return_attention_mask=False, truncation=True, max_length=MAX_LENGTH, padding=True)['input_ids'][:, 1:]
        return prefix, suffix

def run_model(device, df):
    # Load abstract model
    model = ShardedLlama(checkpoint_path, device=f'cuda:{device}')
    n_layers = len(model.layer_names)
    f = partial(get_tokens, tokenizer=model.tokenizer)
    inputs = df.apply(f, axis=1).values
    batches = np.array_split(inputs, N_BATCHES)
    
    # Run model
    # Load each layer only once, run all examples through it,
    # keep activations in memory, and pass them to the next layer.
    for layer_num in range(n_layers):
        # determine which class to load
        layer_class = MidLayer
        if layer_num == 0:
            layer_class = EmbLayer
        elif layer_num == n_layers - 1:
            layer_class = LastLayer
        layer_model = layer_class(checkpoint_path, layer_num, device=f'cuda:{device}')

        # Load layer - slowest operation
        start_time = time.time()
        layer_model.load_layer(layer_model.layer_names[0])
        print(f'Layer {layer_num} loaded in {time.time() - start_time:.3f} s')
        clean_memory()
        
        # Run all examples through this layer and replace batches content with the results
        for batch_idx, batch in enumerate(batches):
            if layer_num == 0:
                batches[batch_idx] = layer_model(batch)
            elif layer_num == n_layers - 1:
                batches[batch_idx] = layer_model(*batch, output_token=4874)
            else:
                batches[batch_idx] = layer_model(*batch)

        # Remove layer
        layer_model.layers[0].to('meta')

    return batches

In [ ]:
if DO_XWIN:
    with ThreadPoolExecutor() as executor:
        outputs = list(executor.map(run_model, [0, 1], np.array_split(df, 2)))
        outputs = sum(outputs, [])
    outputs = np.concatenate(outputs, axis=0)

In [ ]:
clean_memory()

In [ ]:
if os.path.exists("/root/.cache/"):
    shutil.rmtree("/root/.cache/")
checkpoint_path = Path("/root/.cache/")
checkpoint_path.mkdir(exist_ok=True, parents=True)

for part in [1, 2]:
    source_dir = Path(f'/kaggle/input/platypus2-70b-instruct-part{part}')
    for path in source_dir.glob('*'):
        try:
            (checkpoint_path / path.name).symlink_to(path)
        except:
            pass

In [ ]:
if DO_PLATYPUS:
    with ThreadPoolExecutor() as executor:
        outputs_platy = list(executor.map(run_model, [0, 1], np.array_split(df, 2)))
        outputs_platy = sum(outputs_platy, [])
    outputs_platy = np.concatenate(outputs_platy, axis=0)

In [ ]:
preds = torch.argsort(-logits, dim=1).numpy()

In [ ]:
if DO_PLATYPUS and DO_XWIN:
    if ADD_MLM_TO_ENSEMBLE:
        hard_logits = logits[hard_questions].numpy()
        preds[hard_questions] = np.argsort(-((1 - PLATY_WEIGHT - MLM_WEIGHT)*outputs + PLATY_WEIGHT*outputs_platy + MLM_WEIGHT*hard_logits), axis=1)
    else:
        preds[hard_questions] = np.argsort(-((1 - PLATY_WEIGHT)*outputs + PLATY_WEIGHT*outputs_platy), axis=1)
elif DO_PLATYPUS:
    if ADD_MLM_TO_ENSEMBLE:
        hard_logits = logits[hard_questions].numpy()
        preds[hard_questions] = np.argsort(-((1 - MLM_WEIGHT)*outputs_platy + MLM_WEIGHT*hard_logits), axis=1)
    else:
        preds[hard_questions] = np.argsort(-outputs_platy, axis=1)
elif DO_XWIN:
    if ADD_MLM_TO_ENSEMBLE:
        hard_logits = logits[hard_questions].numpy()
        preds[hard_questions] = np.argsort(-((1 - MLM_WEIGHT)*outputs + MLM_WEIGHT*hard_logits), axis=1)
    else:
        preds[hard_questions] = np.argsort(-outputs, axis=1)

In [ ]:
preds_as_letters = np.array(list('ABCDE'))[preds]
preds_as_letters = [' '.join(row) for row in preds_as_letters[:, :3]]

In [ ]:
submission_df = pd.read_csv(test_df_path)
submission_df['prediction'] = preds_as_letters
submission_df = submission_df[['id', 'prediction']]
submission_df.to_csv('submission.csv', index=False)

In [ ]:
print(submission_df)

In [ ]:
df_train = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/train.csv')

In [ ]:
sum([df_train.answer.iloc[idx] == submission_df.prediction.iloc[idx][0] for idx in range(200)])